## 04b - Topic model training

Trains a BERTopic model on the Italian reviews and saves it with the topic of each review.
A new model can number and size its topics differently, so after a run the label map in 04c may need to be rebuilt.

Input: `data/processed/trenord_reviews_clean.csv`

Output: `data/processed/trenord_reviews_topics.csv` and `data/processed/bertopic_model/`

In [1]:
# --- IMPORTS ---
import pandas as pd
from bertopic import BERTopic
from bertopic.representation import MaximalMarginalRelevance
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from spacy.lang.it.stop_words import STOP_WORDS as italian_stopwords

# --- LOAD DATA ---
df = pd.read_csv("../data/processed/trenord_reviews_clean.csv")
df["date"] = pd.to_datetime(df["date"])

print(f"Loaded {len(df)} reviews")
print(f"Language distribution:\n{df['language'].value_counts()}")

W1102 16:54:49.143000 15560 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Loaded 7329 reviews
Language distribution:
language
it         6951
unknown     363
en            5
la            5
fr            2
es            1
mi            1
pt            1
Name: count, dtype: int64


In [2]:
# --- PREPARE TEXTS ---
# Filter for Italian reviews because we want to analyze them separately.
italian_df = df[df["language"] == "it"].copy()

# We use text_clean (not text_lemmatized) because BERTopic uses transformer
# embeddings that work better with natural text. BERTopic handles
# tokenization and stopwords internally.
texts = italian_df["text_clean"].tolist() # tolist() converts the pandas Series to a list of strings

# BERTopic cannot handle empty strings, so we replace them with a single space " ".
# We check two conditions for each text:
# - it must be a string (not NaN)
# - it must contain at least one non-space character
for i in range(len(texts)):
    text = texts[i] # Get the text at index i
    is_valid = isinstance(text, str) and len(text.strip()) > 0
    if not is_valid:
        texts[i] = " "

# Count how many texts ended up empty after the replacement above
empty_count = 0
for text in texts:
    if text.strip() == "":
        empty_count += 1

print(f"Italian reviews : {len(italian_df)}")
print(f"Empty texts     : {empty_count}")
print(f"Texts prepared  : {len(texts)}")

Italian reviews : 6951
Empty texts     : 0
Texts prepared  : 6951


In [3]:
# --- CONFIGURE BERTOPIC ---
# BERTopic works in 4 steps:
# 1. EMBEDDING : converts texts into numerical vectors
# 2. UMAP      : reduces vector dimensions for clustering
# 3. HDBSCAN   : groups similar texts into topics
# 4. c-TF-IDF  : finds the most representative words per topic
# On top of these, we add a representation model (MMR) to refine topic keywords.

# Step 1 - Embedding model
# paraphrase-multilingual-mpnet-base-v2 is the best lightweight
# multilingual model that runs on CPU and handles Italian well
embedding_model = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")

# Step 2 - UMAP for dimensionality reduction
# random_state=42 is critical - without it results change every run
umap_model = UMAP(
    n_neighbors=15,   # how many neighbors each point considers
    n_components=5,   # reduce to 5 dimensions before clustering
    min_dist=0.0,     # pack cluster points tightly together
    metric="cosine",  # best distance metric for text embeddings
    random_state=42   # makes results reproducible
)

# Step 3 - HDBSCAN for clustering
# min_cluster_size=30 means a topic needs at least 30 reviews to be valid
hdbscan_model = HDBSCAN(
    min_cluster_size=30,
    metric="euclidean",
    prediction_data=True  # needed to assign topics to new documents
)

# Step 4 - CountVectorizer with Italian stopwords and bigrams
# We define this here so we use it once in update_topics later
# ngram_range=(1,2) means we consider both single words and two-word phrases
# BERTopic fits the vectorizer on one joined text per topic, so min_df counts topics, not reviews.
# It changes only the keywords, not which reviews belong to which topic (compared in 04a).
vectorizer_model = CountVectorizer(
    stop_words=list(italian_stopwords),  # remove Italian stopwords
    min_df=5,                            # keyword must appear in at least 5 topics
    ngram_range=(1, 2)                   # single words + two-word phrases
)

# Step 5 - Representation model: Maximal Marginal Relevance (MMR)
# By default, c-TF-IDF may return redundant keywords that mean the same thing,
# such as "biglietto", "biglietti", "acquisto biglietti".
# MMR re-ranks the candidate keywords to maximize diversity, so the final
# topic representation contains less redundant and more informative words.
# diversity ranges from 0 (no diversity) to 1 (maximum diversity).
# We use 0.3 - the value recommended in the official BERTopic documentation -
# which reduces redundancy while keeping the keywords relevant to the topic.
representation_model = MaximalMarginalRelevance(diversity=0.3)

# Combine everything into BERTopic
topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,        # Italian stopwords and bigrams
    representation_model=representation_model, # MMR for diverse topic keywords
    top_n_words=10,
    language="multilingual",
    calculate_probabilities=False,  # faster, we don't need soft assignments
    verbose=True
)

print("BERTopic configured and ready.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   6%|▌         | 12/199 [00:00<00:02, 85.86it/s]

Loading weights:  11%|█         | 22/199 [00:00<00:04, 44.16it/s]

Loading weights:  14%|█▍        | 28/199 [00:00<00:06, 28.31it/s]

Loading weights:  16%|█▌        | 32/199 [00:00<00:06, 27.51it/s]

Loading weights:  18%|█▊        | 36/199 [00:01<00:05, 28.32it/s]

Loading weights:  20%|██        | 40/199 [00:01<00:10, 14.88it/s]

Loading weights:  57%|█████▋    | 113/199 [00:01<00:00, 100.85it/s]

Loading weights:  68%|██████▊   | 136/199 [00:02<00:00, 113.33it/s]

Loading weights: 100%|██████████| 199/199 [00:02<00:00, 97.27it/s] 

BERTopic configured and ready.


In [4]:
# --- TRAIN BERTOPIC ---
# fit_transform() trains the model AND assigns a topic to each text.
# topic -1 means "outlier" - a review that doesn't fit any topic clearly.

print("Training BERTopic... this may take several minutes on CPU.")

# We ignore the second returned value (probabilities) because we set
# calculate_probabilities=False, so it would be None anyway.
topics, _ = topic_model.fit_transform(texts)

# Store the topic assignments in the dataframe
italian_df["topic"] = topics

# Count how many reviews fall into each topic.
# value_counts() returns a Series: index = topic id, value = number of reviews.
topic_counts = italian_df["topic"].value_counts()

# Count outliers - reviews assigned to the special topic -1
n_outliers = topic_counts.get(-1, 0)  # .get avoids an error if there are no outliers

# Count real topics - all unique topic ids except the outlier topic -1
unique_topics = set(topics)
unique_topics.discard(-1)             # remove -1 if present; does nothing if absent
n_topics = len(unique_topics)

# Count assigned reviews - total reviews minus the outliers
n_assigned = len(topics) - n_outliers

print(f"\nTraining complete!")
print(f"Topics found : {n_topics}")
print(f"Outliers     : {n_outliers}")
print(f"Assigned     : {n_assigned}")

2026-11-02 16:55:05,009 - BERTopic - Embedding - Transforming documents to embeddings.


Training BERTopic... this may take several minutes on CPU.


Batches:   0%|          | 0/218 [00:00<?, ?it/s]

Batches:   0%|          | 1/218 [00:04<17:14,  4.77s/it]

Batches:   1%|          | 2/218 [00:08<15:55,  4.42s/it]

Batches:   1%|▏         | 3/218 [00:13<15:52,  4.43s/it]

... (220 outputs left out)

2026-11-02 17:00:38,975 - BERTopic - Cluster - Completed ✓


2026-11-02 17:00:38,982 - BERTopic - Representation - Fine-tuning topics using representation models.


2026-11-02 17:00:51,175 - BERTopic - Representation - Completed ✓



Training complete!
Topics found : 30
Outliers     : 1601
Assigned     : 5350


In [5]:
# --- REDUCE OUTLIERS ---
# We assign each outlier to its closest topic using embedding similarity.
# strategy="embeddings" : uses cosine similarity between review and topic centroid
# threshold=0.0 : assigns ALL outliers - we inspected them and they are valid reviews

# Count outliers before reduction, straight from the current topic column
outliers_before = (italian_df["topic"] == -1).sum()

# Reassign outliers to their closest topic
new_topics = topic_model.reduce_outliers(
    texts,
    topics,
    strategy="embeddings",
    threshold=0.0
)
italian_df["topic"] = new_topics

# Count outliers after reduction - should be 0 since threshold=0.0 assigns all of them
outliers_after   = (italian_df["topic"] == -1).sum()
reviews_assigned = (italian_df["topic"] != -1).sum()

print(f"Outliers before : {outliers_before}")
print(f"Outliers after  : {outliers_after}")
print(f"Reviews assigned: {reviews_assigned}")

Outliers before : 1601
Outliers after  : 0
Reviews assigned: 6951


In [6]:
# --- UPDATE TOPIC REPRESENTATIONS ---
# After reducing outliers, we recompute the topic representations so they
# reflect the new assignments. We pass two models explicitly:
# 1. vectorizer_model     : applies Italian stopwords and bigrams
# 2. representation_model : applies MMR to keep topic keywords diverse
# Passing both here ensures the final representations are consistent with
# the configuration we set when building the model.

topic_model.update_topics(
    texts,
    topics=new_topics,
    vectorizer_model=vectorizer_model,
    representation_model=representation_model  # MMR - keeps keywords diverse
)

# Retrieve the updated topic information
topic_info = topic_model.get_topic_info()

# Keep only real topics - exclude the outlier topic -1
real_topics = topic_info[topic_info["Topic"] != -1]

print(f"Total topics : {len(real_topics)}")
print()
for _, row in real_topics.iterrows():
    print(f"Topic {row['Topic']:3d} | {row['Count']:4d} reviews | {row['Name']}")

2026-11-02 17:02:06,312 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


Total topics : 30

Topic   0 | 1214 reviews | 0_treni_treno_ritardi_app
Topic   1 | 1143 reviews | 1_biglietti_acquistare biglietti_app_acquisto
Topic   2 |  641 reviews | 2_pagamento_abbonamenti_rinnovare abbonamento_credito
Topic   3 |  499 reviews | 3_lenta_inutile_inaffidabile_difficile
Topic   4 |  515 reviews | 4_app_aggiornamento_telefono_android
Topic   5 |  219 reviews | 5_login_account_accesso_errore
Topic   6 |  185 reviews | 6_ritardi_ritardo_ritardi cancellazioni_arrivano ritardo
Topic   7 |  173 reviews | 7_stazioni_stazione_stazioni preferite_aggiornamento
Topic   8 |  176 reviews | 8_ottima app_app comoda_app facile_app funziona
Topic   9 |  152 reviews | 9_ottima applicazione_applicazione_utilissima_applicazione utile
Topic  10 |  164 reviews | 10_ottima_perfetta_perfetto_fantastica
Topic  11 |  145 reviews | 11_facile_semplice_facile usare_intuitivo
Topic  12 |  175 reviews | 12_biglietti_acquisto biglietti_acquistare biglietti_app
Topic  13 |  118 reviews | 13_biglie

In [7]:
# --- SAVE MODEL AND DATA ---
# We save the BERTopic model in safetensors format - safer and more compact than pickle.
# We save the dataframe to a NEW file (not the cleaning output) because this notebook
# adds the topic assignments. Keeping outputs separate makes the pipeline traceable:
# - 02_cleaning.ipynb  → trenord_reviews_clean.csv   (cleaned text, no topics)
# - 04b_topic_modeling_training.ipynb → trenord_reviews_topics.csv  (adds topic assignments)

# Save BERTopic model in safetensors format
embedding_model_name = "paraphrase-multilingual-mpnet-base-v2"

topic_model.save(
    "../data/processed/bertopic_model",
    serialization="safetensors",
    save_ctfidf=True,
    save_embedding_model=embedding_model_name
)
print("BERTopic model saved in safetensors format.")

# Save dataframe with topic IDs to a dedicated file - labels are assigned later in 04c
italian_df.to_csv(
    "../data/processed/trenord_reviews_topics.csv",
    index=False,
    encoding="utf-8-sig"
)
print(f"Dataframe saved: {len(italian_df):,} reviews with topic assignments.")

BERTopic model saved in safetensors format.
Dataframe saved: 6,951 reviews with topic assignments.
